In [104]:
import pyspark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import types

In [105]:
df2 = pd.read_parquet("yellow_tripdata_2025-11.parquet")

In [106]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [107]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")


In [108]:
df2 = pd.read_parquet("yellow_tripdata_2025-11.parquet")

In [164]:
#df.repartition(4).write.parquet("yellow_tripdata_2025-11_2.parquet")

In [110]:
df3 = spark.read.parquet('yellow_tripdata_2025-11_2.parquet')

In [111]:
df3.show()
df3.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-04 21:00:59|  2025-11-04 21:09:12|              1|          0.0|         1|                 N|         162|    

In [112]:
df3.registerTempTable('nov_trips')

/home/clairma/Desktop/1-Lemonde/TECH/0-mes-exos-mon-code/DE/Data-Engineering-Zoomcamp/module6/.venv/lib/python3.11/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [113]:
# Question 3
#How many taxi trips were there on the 15th of November?

spark.sql("""
SELECT COUNT(*)
FROM nov_trips
WHERE
tpep_pickup_datetime >= '2025-11-15' AND
tpep_pickup_datetime < '2025-11-16'
LIMIT 100;
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [142]:
spark.sql("""
SELECT trip_distance, tpep_dropoff_datetime, tpep_pickup_datetime, timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime) AS length
from nov_trips
ORDER by length DESC
limit 1
""").show()

[Stage 65:=============================>                            (4 + 4) / 8]

+-------------+---------------------+--------------------+------+
|trip_distance|tpep_dropoff_datetime|tpep_pickup_datetime|length|
+-------------+---------------------+--------------------+------+
|       121.17|  2025-11-30 15:01:00| 2025-11-26 20:22:12|    90|
+-------------+---------------------+--------------------+------+



In [144]:
df4= spark.read.csv('taxi_zone_lookup.csv')

In [147]:
df5=pd.read_csv('taxi_zone_lookup.csv')

In [150]:
df5.dtypes

LocationID      int64
Borough           str
Zone              str
service_zone      str
dtype: object

In [151]:
spark.createDataFrame(df5).schema

StructType([StructField('LocationID', LongType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [156]:
schema = types.StructType([types.StructField('LocationID', types.LongType(), True), 
            types.StructField('Borough', types.StringType(), True), 
            types.StructField('Zone', types.StringType(), True), 
            types.StructField('service_zone', types.StringType(), True)])

In [158]:
df6 = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('taxi_zone_lookup.csv')

In [160]:
df6.registerTempTable('zones')

In [162]:
spark.sql("""
SELECT COUNT(*)
FROM nov_trips
LIMIT 10;
""").show()

+--------+
|count(1)|
+--------+
| 4181444|
+--------+



In [163]:
spark.sql("""
SELECT COUNT(*)
FROM zones
LIMIT 10;
""").show()

+--------+
|count(1)|
+--------+
|     265|
+--------+



In [183]:
spark.sql("""
SELECT z.Zone, COUNT(*) as total_amount
FROM zones z, nov_trips n
WHERE z.LocationID = n.PUlocationID
group by z.Zone
order by total_amount ASC
;
""").show()

[Stage 102:============================>                            (4 + 4) / 8]

+--------------------+------------+
|                Zone|total_amount|
+--------------------+------------+
|Eltingville/Annad...|           1|
|       Arden Heights|           1|
|Governor's Island...|           1|
|       Port Richmond|           3|
|         Great Kills|           4|
| Green-Wood Cemetery|           4|
|       Rikers Island|           4|
|   Rossville/Woodrow|           4|
|         Jamaica Bay|           5|
|         Westerleigh|          12|
|       West Brighton|          14|
|New Dorp/Midland ...|          14|
|             Oakwood|          14|
|        Crotona Park|          14|
|       Willets Point|          15|
|Breezy Point/Fort...|          16|
|Saint George/New ...|          17|
|       Broad Channel|          18|
|     Mariners Harbor|          21|
|Heartland Village...|          22|
+--------------------+------------+
only showing top 20 rows
